In [ ]:
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# INPUT
# ============================================================

census_csv = (
    r"F:\vietpop\vietpop-neuralnetwork\pomelo_input_data\VNM\vnm_wpop_census.csv"
)

features_csv = (
    r"F:\vietpop\vietpop_dm\prj_vn_2019\outputs\rf_pixel_train\features.csv"
)

commune_raster = (
    r"F:\vietpop\vietpop_dm\prj_vn_2019\data\VNM_covariates_2019"
    r"\mastergrid_commune2019_masked.tif"
)

district_raster = (
    r"F:\vietpop\vietpop_dm\prj_vn_2019\data\VNM_covariates_2019"
    r"\mastergrid_district2019_masked.tif"
)

output_dir = Path(
    r"F:\vietpop\vietpop_dm\prj_vn_2019\data\VNM_covariates_2019"
)

output_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# OUTPUT
# ============================================================

commune_dens_out = output_dir / "commune_density.tif"
commune_area_out = output_dir / "commune_area.tif"

district_dens_out = output_dir / "district_density.tif"
district_area_out = output_dir / "district_area.tif"


# ============================================================
# 1. READ CENSUS
# ============================================================

census = pd.read_csv(census_csv)

print("Census:")
print(census.head())
print(census.columns.tolist())


# Chỉ lấy các cột cần thiết
census = census[["SID", "P_2019", "GR_SID"]].copy()

census["SID"] = pd.to_numeric(census["SID"], errors="coerce")
census["GR_SID"] = pd.to_numeric(census["GR_SID"], errors="coerce")
census["P_2019"] = pd.to_numeric(census["P_2019"], errors="coerce")

census = census.dropna(subset=["SID", "GR_SID", "P_2019"])

# Nếu SID có dạng số nguyên
census["SID"] = census["SID"].astype(np.int64)
census["GR_SID"] = census["GR_SID"].astype(np.int64)


# ============================================================
# 2. READ FEATURES
# ============================================================

features = pd.read_csv(features_csv)

print("\nFeatures:")
print(features.head())
print(features.columns.tolist())


# id = commune ID
features["id"] = pd.to_numeric(features["id"], errors="coerce")

# dens từ features
features["dens"] = pd.to_numeric(features["dens"], errors="coerce")

features = features.dropna(subset=["id", "dens"])

features["id"] = features["id"].astype(np.int64)


# ============================================================
# 3. MERGE CENSUS + FEATURES
# ============================================================

df = census.merge(
    features[["id", "dens"]],
    left_on="SID",
    right_on="id",
    how="left"
)

print("\nAfter merge:")
print(df[["SID", "P_2019", "GR_SID", "dens"]].head())

print(
    "Missing dens:",
    df["dens"].isna().sum(),
    "/",
    len(df)
)


# ============================================================
# 4. CALCULATE COMMUNE AREA
# ============================================================

# area = population / density
df["area"] = np.nan

valid = (
    df["P_2019"].notna()
    & df["dens"].notna()
    & (df["dens"] > 0)
)

df.loc[valid, "area"] = (
    df.loc[valid, "P_2019"]
    / df.loc[valid, "dens"]
)

print("\nCommune area:")
print(df["area"].describe())


# ============================================================
# 5. DISTRICT AGGREGATION
# ============================================================

district = (
    df.groupby("GR_SID", as_index=False)
      .agg(
          pop_district=("P_2019", "sum"),
          area_district=("area", "sum")
      )
)

# District density
district["dens_district"] = (
    district["pop_district"]
    / district["area_district"]
)

print("\nDistrict:")
print(district.head())

print(
    "\nNumber of districts:",
    len(district)
)


# ============================================================
# 6. CREATE LOOKUP DICTIONARIES
# ============================================================

# ----------------------------
# Commune
# ----------------------------

commune_dens_dict = dict(
    zip(
        df["SID"],
        df["dens"]
    )
)

commune_area_dict = dict(
    zip(
        df["SID"],
        df["area"]
    )
)


# ----------------------------
# District
# ----------------------------

district_dens_dict = dict(
    zip(
        district["GR_SID"],
        district["dens_district"]
    )
)

district_area_dict = dict(
    zip(
        district["GR_SID"],
        district["area_district"]
    )
)


# ============================================================
# 7. FUNCTION: CREATE RASTER FROM LOOKUP
# ============================================================

def create_raster_from_lookup(
    input_raster,
    output_raster,
    lookup,
    dtype="float32",
    nodata=-9999.0
):
    print(f"\nProcessing:\n{input_raster}\n-> {output_raster}")

    with rasterio.open(input_raster) as src:
        data = src.read(1)
        profile = src.profile.copy()
        src_nodata = src.nodata

        valid_mask = (data != src_nodata) if src_nodata is not None else np.ones(data.shape, dtype=bool)

        # ID âm hoặc không hợp lệ -> loại khỏi valid_mask trước khi index
        max_id = int(data[valid_mask].max()) if valid_mask.any() else 0

        # Bảng tra cứu dạng mảng: index = ID, value = giá trị cần gán
        lut = np.full(max_id + 1, nodata, dtype=np.float32)
        found = 0
        for region_id, value in lookup.items():
            if 0 <= region_id <= max_id and value is not None and np.isfinite(value):
                lut[region_id] = value
                found += 1

        # Clip ID để tránh out-of-range khi index (pixel invalid sẽ bị ghi đè lại bằng nodata ngay sau)
        safe_ids = np.clip(data, 0, max_id)
        out = lut[safe_ids]
        out[~valid_mask] = nodata

        print("Matched IDs:", found)
        print("Missing IDs:", len(np.unique(data[valid_mask])) - found)

        profile.update(dtype=dtype, count=1, nodata=nodata, compress="lzw")

        with rasterio.open(output_raster, "w", **profile) as dst:
            dst.write(out.astype(dtype), 1)

    print("Saved:", output_raster)


# ============================================================
# 8. COMMUNE DENSITY
# ============================================================

create_raster_from_lookup(
    input_raster=commune_raster,
    output_raster=commune_dens_out,
    lookup=commune_dens_dict
)


# ============================================================
# 9. COMMUNE AREA
# ============================================================

create_raster_from_lookup(
    input_raster=commune_raster,
    output_raster=commune_area_out,
    lookup=commune_area_dict
)


# ============================================================
# 10. DISTRICT DENSITY
# ============================================================

create_raster_from_lookup(
    input_raster=district_raster,
    output_raster=district_dens_out,
    lookup=district_dens_dict
)


# ============================================================
# 11. DISTRICT AREA
# ============================================================

create_raster_from_lookup(
    input_raster=district_raster,
    output_raster=district_area_out,
    lookup=district_area_dict
)


# ============================================================
# DONE
# ============================================================

print("\n========================================")
print("DONE")
print("========================================")

print("Commune density :", commune_dens_out)
print("Commune area    :", commune_area_out)
print("District density:", district_dens_out)
print("District area   :", district_area_out)

In [ ]:
import os
import rasterio
import numpy as np

# Input rasters
base_dir = r"F:\vietpop\vietpop_dm\prj_vn_2019\data\VNM_covariates_2019"

files = {
    "building_count": os.path.join(base_dir, "building_counts_2019_100m.tif"),
    "built_s": os.path.join(base_dir, "vnm_built_S_GHS_U_wFGW_100m_v1_2019.tif"),
    "built_v": os.path.join(base_dir, "vnm_built_V_GHS_U_wFGW_100m_v1_2019.tif"),
    "nightlights": os.path.join(base_dir, "vnm_viirs_nvf_2019_100m_v1.tif"),
    "elevation": os.path.join(base_dir, "vnm_elevation_merit103_100m_v1.tif"),
    "highway_dist": os.path.join(base_dir, "vnm_highway_dist_osm_2023_100m_v1.tif"),
    "road_intr_dist": os.path.join(base_dir, "vnm_rd_intrs_dist_osm_2023_100m_v1.tif"),
    "waterbodies_dist": os.path.join(base_dir, "vnm_waterbodies_dist_osm_2023_100m_v1.tif"),
    "inland_water_dist": os.path.join(base_dir, "vnm_dist_inland_water_100m_esa_2021_v1.tif"),
    "coastline_dist": os.path.join(base_dir, "vnm_coastline_dst_100m_v1.tif"),
    "wdpa_dist": os.path.join(base_dir, "vnm_WDPA_pre2019_cat1_dist_100m_v1.tif"),
    "esalc_11": os.path.join(base_dir, "vnm_esalc_11_dst_2019_100m_v1.tif"),
    "esalc_40": os.path.join(base_dir, "vnm_esalc_40_dst_2019_100m_v1.tif"),
    "esalc_130": os.path.join(base_dir, "vnm_esalc_130_dst_2019_100m_v1.tif"),
    "esalc_140": os.path.join(base_dir, "vnm_esalc_140_dst_2019_100m_v1.tif"),
    "esalc_150": os.path.join(base_dir, "vnm_esalc_150_dst_2019_100m_v1.tif"),
    "esalc_160": os.path.join(base_dir, "vnm_esalc_160_dst_2019_100m_v1.tif"),
    "esalc_190": os.path.join(base_dir, "vnm_esalc_190_dst_2019_100m_v1.tif"),
    "esalc_200": os.path.join(base_dir, "vnm_esalc_200_dst_2019_100m_v1.tif"),
    "bsgme": os.path.join(base_dir, "vnm_bsgme_v0a_100m_2019_resampled.tif"),
    "bsgme_dst": os.path.join(base_dir, "vnm_dst_bsgme_100m_2019_resampled.tif"),
    "slope": os.path.join(base_dir, "vnm_slope_merit103_100m_v1.tif"),
    "inland_water": os.path.join(base_dir, "vnm_inland_water_BIN_100m_esa_2021_v1.tif"),

    "commune_id": os.path.join(base_dir, "mastergrid_commune2019_masked.tif"),
    "commune_area": os.path.join(base_dir, "commune_area.tif"),
    "commune_dens": os.path.join(base_dir, "commune_density.tif"),

    "district_id": os.path.join(base_dir, "mastergrid_district2019_masked.tif"),
    "district_area": os.path.join(base_dir, "district_area.tif"),
    "district_dens": os.path.join(base_dir, "district_density.tif"),
}

# Output
output = r"F:\vietpop\gnn\Shadamap-main\Shadamap-main\data_lib\dataset.tif"

# Check files
print(f"Number of rasters: {len(files)}\n")

for name, path in files.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Không tìm thấy raster:\n{name}: {path}")
    print(f"OK: {name}")

# Reference raster
first_name = next(iter(files))
first_file = files[first_name]

with rasterio.open(first_file) as src:
    profile = src.profile.copy()
    height = src.height
    width = src.width
    crs = src.crs
    transform = src.transform
    res = src.res

print("\nReference raster:")
print("  Name       :", first_name)
print("  Shape      :", height, width)
print("  CRS        :", crs)
print("  Resolution :", res)

# Output profile
profile.update(
    count=len(files),
    dtype="float32",
    compress="deflate",
    predictor=2,
    zlevel=6,
    nodata=-9999
)

# Stack rasters
with rasterio.open(output, "w", **profile) as dst:
    for band_idx, (name, file) in enumerate(files.items(), start=1):
        print(f"\n[{band_idx:02d}/{len(files):02d}] {name}")

        with rasterio.open(file) as src:
            if src.height != height or src.width != width:
                raise ValueError(
                    f"Shape không khớp: {name}\n"
                    f"Expected: {(height, width)}\n"
                    f"Got     : {(src.height, src.width)}"
                )

            if src.crs != crs:
                raise ValueError(
                    f"CRS không khớp: {name}\n"
                    f"Expected: {crs}\n"
                    f"Got     : {src.crs}"
                )

            if not np.allclose(
                np.array(src.transform),
                np.array(transform),
                rtol=0,
                atol=1e-6
            ):
                raise ValueError(
                    f"Transform không khớp: {name}\n"
                    f"Expected: {transform}\n"
                    f"Got     : {src.transform}"
                )

            data = src.read(1).astype("float32")

            if src.nodata is not None:
                data[data == src.nodata] = 0

            data = np.nan_to_num(
                data,
                nan=0.0,
                posinf=0.0,
                neginf=0.0
            )

            dst.write(data, band_idx)
            dst.set_band_description(band_idx, name)

print("\nDONE")
print("Output:", output)
print("Bands :", len(files))

print("\nBand order:")
for i, name in enumerate(files.keys(), start=1):
    print(f"{i:02d}: {name}")

In [2]:
import pandas as pd
census = pd.read_csv(r'vnm_wpop_census.csv')
commune_data = census.sort_values('SID')

commune_pop = (commune_data['P_2019'] / commune_data['area']).to_numpy()
commune_area = commune_data['area'].to_numpy()


district_data = (
    census.groupby('GR_SID')[['P_2019', 'area']]
    .sum()
    .sort_index()
)

district_pop = (
    district_data['P_2019'] / district_data['area']
).to_numpy()

In [ ]:
district_data[]

,P_2019,area
GR_SID,,
1,0,56594
2,213434,1043
3,131908,552
4,156604,1972
5,315468,5781
...,...,...
707,135224,9241
708,174056,19447
709,56052,22312
